In [1]:
from local_functions import pix_to_deg,read_data,calculate_bf,plot_bfs,shrink_axes,make_subplots, inset_axes
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns


# A4 dimensions in inches
a4_width = 8.27 
a4_height = 11.69 
subj_colors = sns.color_palette("Spectral",21)

CFFI extension type
  ABI: PRESENT
  API: PRESENT
  Environment variable: RPY2_CFFI_MODE=ANY
  Loaded: CFFI_MODE.API
rpy2 version:
rpy2.rinterface: 3.6.3
Python version:
3.14.2 | packaged by conda-forge | (main, Dec  6 2025, 11:34:40) [Clang 20.1.8 ]
Looking for R's HOME:
  Environment variable R_HOME: None
  Calling `R RHOME`: /opt/anaconda3/envs/py-r-bayes/lib/R
  Environment variable R_LIBS_USER: None
R's value for LD_LIBRARY_PATH:
  
R version:
  In the PATH: R version 4.5.2 (2025-10-31) -- "[Not] Part in a Rumble"
  Loading R library from rpy2: OK
Additional directories to load R packages from:
  None
C extension compilation:
  include:
    ['/opt/anaconda3/envs/py-r-bayes/lib/R/include']
  libraries:
    ['R', 'bz2', 'z', 'dl', 'm', 'iconv', 'icuuc', 'icui18n']
  library_dirs:
    ['/opt/anaconda3/envs/py-r-bayes/lib', '/opt/anaconda3/envs/py-r-bayes/lib/R/lib', '/opt/anaconda3/envs/py-r-bayes/lib']
  extra_compile_args:
    ['-std=c99']
  extra_link_args:
    ['-fopenmp', '-Wl,-

### read data

In [2]:
task = 'exp1b'
bids_dir = '/Volumes/LaCie2/2021_Occlusion_behavioural/data'
subjids = [i for i in np.arange(1,22)]
all_epochs = read_data(bids_dir,subjids,task=task)

Reading /Volumes/LaCie2/2021_Occlusion_behavioural/data/exp1b/derivatives/preprocessed/sub-S01_exp1b_eyetracking-preproc-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =    -500.00 ...    3300.00 ms
        0 CTF compensation matrices available
Adding metadata with 34 columns
252 matching events found
No baseline correction applied
0 projection items activated
Reading /Volumes/LaCie2/2021_Occlusion_behavioural/data/exp1b/derivatives/preprocessed/sub-S02_exp1b_eyetracking-preproc-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =    -500.00 ...    3300.00 ms
        0 CTF compensation matrices available
Adding metadata with 34 columns
252 matching events found
No baseline correction applied
0 projection items activated
Reading /Volumes/LaCie2/2021_Occlusion_behavioural/data/exp1b/derivatives/preprocessed/sub-S03_exp1b_eyetracking-preproc-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =    -500.00 ...    3300.00 ms


### make plots

In [9]:
### SETUP
%matplotlib qt
plt.close('all')
fig,axs = plt.subplots(3,3,figsize=(a4_width,a4_height*0.9))

# Labels for the subplots
dat = all_epochs[0].metadata.copy()
speed_labels = [f"{np.round(dat.loc[dat.speed_idx==s,'velocity'].unique()[0],2)}\u00B0" for s in range(3)]
speed_titles = ['fast','medium','slow']
for i, (ax, label) in enumerate(zip([axs[0][0],axs[1][0],axs[2][0]],['(a)','(b)','(c)'])):
    pos = ax.get_position()  
    x = pos.x0 - 0.05
    y = pos.y1 -0.01
    fig.text(x, y, label, fontsize=12, fontweight='bold', va='top', ha='left')


import seaborn as sns
colors = sns.color_palette('mako',7)

#************************************
### (A) BEHAVIOURAL PERFORMANCE
#************************************

group_data = []
for s in range(3):
    ax = shrink_axes(axs[0][s],percent_Y=0.7)

    group_data_speed = []

    for subj in range(len(all_epochs)):
        dat = all_epochs[subj].metadata.copy()
        # ignoring different motion directions [L->R and R->L]
        dat.loc[dat.direction==2,'stop_pos_deg'] = dat.loc[dat.direction==2,'stop_pos_deg']*-1
        dat.loc[dat.direction==2,'stop_pos'] = dat.loc[dat.direction==2,'stop_pos']*-1
        dat.loc[dat.direction==2,'pos_pressed_deg'] = dat.loc[dat.direction==2,'pos_pressed_deg']*-1
        dat.loc[dat.direction==2,'pos_pressed'] = dat.loc[dat.direction==2,'pos_pressed']*-1


        dat_curr = dat.loc[dat['speed_idx']==s,:]
        # x = np.sort(dat_curr['stop_movement_time'].unique())
        # print(dat_curr.groupby('stop_movement_time').mean(numeric_only=True))
        # y = dat_curr.groupby('stop_movement_time').mean(numeric_only=True).pos_pressed_deg.to_numpy()

        xy = dat_curr.groupby('stop_movement_time').mean(numeric_only=True)


        ax.plot(xy.index,xy.pos_pressed_deg,'k.',alpha=0.5)
        
        group_data_speed.append(xy)
        

    true_y = xy.stop_pos_deg.to_numpy()
    [ax.plot(xy.index[i],true_y[i],marker='*',color=colors[i],alpha=0.6, markersize=10,lw=0.5,markeredgecolor=None) for i in range(7)]

    # style
    ax.set_ylim([-5,8])
    ax.set_yticks(np.arange(-3,8,2))
    ax.set_xticks(xy.index)
    ax.set_xticklabels('')
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    if s == 0:
        label = ax.set_ylabel('Relative position\n clicked (dva)')
        label.set_position((label.get_position()[0],label.get_position()[1]*1.2))
    ax.set_title(f'{speed_titles[s]}\n({speed_labels[s]}per sec)')
    
    # add Bayes factors
    ax_in = inset_axes(ax,
                    width="100%",  # Width of inset as a percentage of axs[1][0] width
                    height="20%",  # Height of inset as a percentage of axs[1][0] height
                    bbox_to_anchor=(0.0, -0.3, 1.0, 1.0),  # Relative position below axs[1][0]
                    bbox_transform=ax.transAxes,  # Transform to use axs[1][0]'s coordinate system
                    loc='lower center') 

    mean_beh_error = np.array([dat.pos_pressed_deg.to_numpy() for dat in group_data_speed])-true_y
    bf_out_beh = calculate_bf(mean_beh_error)
    plot_bfs(fig,ax_in,x=xy.index,y=bf_out_beh,n_cols = 500,bf_cmap="coolwarm",exponential_minmax=18)
    ax_in.set_xlabel('time since trial start (s)')
    ax_in.set_xticklabels([np.around(i,2) for i in xy.index],fontsize=7)
    [ax_in.spines[i].set_visible(False) for i in ['top','right']]

    if s == 0:
        ax_in.set_ylabel('BF',fontsize=12)
    
    group_data.append(group_data_speed)

    ax_in.tick_params(axis='y', which='major', labelsize=6)

    print(f'(a): speed {speed_titles[s]} -- BFs 10^{np.log10(bf_out_beh)}')
    


#************************************
### (B) EYETRACKING
#************************************
group_final_gaze=[]
for s in range(3):
    ax = shrink_axes(axs[1][s],percent_Y=0.8)

    # get trajectory and timing for this current speed
    sample_p = 2
    epos = all_epochs[sample_p].copy()
    epos_curr_speed = epos[epos.metadata['speed_idx']==s]

    x_idx = np.where([i=='xpos' for i in epos_curr_speed.ch_names])[0][0]
    epos_curr_speed._data[epos_curr_speed.metadata.direction==2,x_idx,:] = epos_curr_speed._data[epos_curr_speed.metadata.direction==2,x_idx,:]*-1

    unique_stop_movement_time  = np.sort(epos_curr_speed.metadata.stop_movement_time.unique())

    traj = np.linspace(np.min(epos_curr_speed.metadata.start_pos_deg.unique()),np.max(epos_curr_speed.metadata.start_pos_deg.unique()),epos_curr_speed.metadata['n_frames'].unique()[0])
    total_traj_time = epos_curr_speed.metadata['n_frames'].unique()[0]*1/epos_curr_speed.metadata['monitor_refreshrate'].unique()[0]
    traj_tv = np.arange(0,total_traj_time,1/epos_curr_speed.metadata['monitor_refreshrate'].unique()[0])

    max_time = np.sort(epos_curr_speed.metadata.reappearance_time.unique())[0]
    max_time_idx = np.argmin([np.abs(i-max_time) for i in epos_curr_speed.times])
        

    ax.plot(traj_tv,traj,'k--',label='object trajectory',lw=1)


    for idx,v in enumerate(unique_stop_movement_time):
        toplot = []
        for p in range(len(all_epochs)):
            epos = all_epochs[p].copy()
            if np.all(np.isnan(epos._data)):
                continue

            epos_curr_speed = epos[epos.metadata['speed_idx']==s]
            x_idx = np.where([i=='xpos' for i in epos_curr_speed.ch_names])[0][0]
            epos_curr_speed._data[epos_curr_speed.metadata.direction==2,x_idx,:] = epos_curr_speed._data[epos_curr_speed.metadata.direction==2,x_idx,:]*-1

            sel_epo = epos_curr_speed[(epos_curr_speed.metadata.stop_movement_time==v)]
            toplot.append(np.nanmean(sel_epo._data[:,x_idx],axis=0))
        

        
        # plot mean x-gaze 
        stopping_idx_traj = np.argmin(np.abs(traj_tv-v))
        stopping_idx_epo = (epos.times<=0).sum()+np.argmin(np.abs(epos.times[(epos.times>0)]-traj_tv[stopping_idx_traj]))
        ax.plot(sel_epo.times[0:stopping_idx_epo],np.mean(toplot,axis=0)[0:stopping_idx_epo],color = colors[idx],lw=1,alpha=0.8)
     
    group_final_gaze_speed=[]
    for idx,v in enumerate(unique_stop_movement_time):   
        stopping_idx_traj = np.argmin(np.abs(traj_tv-v))
        stopping_idx_epo = (epos.times<=0).sum()+np.argmin(np.abs(epos.times[(epos.times>0)]-traj_tv[stopping_idx_traj]))
        ax.plot(traj_tv[stopping_idx_traj],traj[stopping_idx_traj],marker='*',color=colors[idx],markeredgecolor=None,alpha=0.6,markersize=10)
        ax.plot(sel_epo.times[stopping_idx_epo],np.mean(toplot,axis=0)[stopping_idx_epo],'.',color = colors[idx],lw=1,alpha=1)
        group_final_gaze_speed.append(np.array(toplot)[:,stopping_idx_epo])

    group_final_gaze.append(group_final_gaze_speed)
    # style axis
    ax.vlines(epos_curr_speed.metadata['occlusion_start_time'].unique()[0],-4.5,4.5,color='k')
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    
    # ax.set_title(f'{speed_titles[s]}\n({speed_labels[s]}per sec)')
    ax.set_xlim([0,sel_epo.times[max_time_idx]])
    ax.set_xlabel('time since trial start (s)')
    
    if s == 0:
        ax.set_ylabel('x-gaze position (dva)')

#************************************
### (C) EYETRACKING & BEHAVIOUR CORRELATION
#************************************
import pandas as pd

for s in range(3):
    all_corr = []
    ax = shrink_axes(axs[2][s],percent_Y=0.8)

    for p in range(len(all_epochs)):
        epos_ppt = all_epochs[p].copy()
        epos = epos_ppt[epos_ppt.metadata['speed_idx']==s]

        if np.all(np.isnan(epos._data)):
            continue

        # ignoring different motion directions [L->R and R->L]
        epos.metadata.loc[epos.metadata.direction==2,'stop_pos_deg'] = epos.metadata.loc[epos.metadata.direction==2,'stop_pos_deg']*-1
        epos.metadata.loc[epos.metadata.direction==2,'stop_pos'] = epos.metadata.loc[epos.metadata.direction==2,'stop_pos']*-1
        epos.metadata.loc[epos.metadata.direction==2,'pos_pressed_deg'] = epos.metadata.loc[epos.metadata.direction==2,'pos_pressed_deg']*-1
        epos.metadata.loc[epos.metadata.direction==2,'pos_pressed'] = epos.metadata.loc[epos.metadata.direction==2,'pos_pressed']*-1
        epos._data[epos.metadata.direction==2,x_idx,:] = epos._data[epos.metadata.direction==2,x_idx,:]*-1

        # calculate behavioural error
        beh_error = epos.metadata.pos_pressed_deg-epos.metadata.stop_pos_deg

        # figuring out timepoints when the object stopped moving / sound stopped playing
        total_traj_time = epos.metadata['n_frames'].unique()[0]*1/epos.metadata['monitor_refreshrate'].unique()[0]
        traj_tv = np.arange(0,total_traj_time,1/epos.metadata['monitor_refreshrate'].unique()[0])
        stopping_idx_traj = [np.argmin(np.abs(traj_tv-v)) for v in epos.metadata.stop_movement_time]
        stopping_idx_epo = [(epos.times<=0).sum()+np.argmin(np.abs(epos.times[(epos.times>0)]-traj_tv[v])) for v in stopping_idx_traj]

        # calculate eyetracking error
        x_idx = np.where([i=='xpos' for i in epos.ch_names])[0][0]
        eye_error = [epos._data[i,x_idx,v] for i,v in enumerate(stopping_idx_epo)]-epos.metadata.stop_pos_deg.to_numpy()

        # plot
        ax.plot(beh_error,eye_error,marker='.',color=subj_colors[p],lw=0,alpha=0.4,markeredgecolor=None)

        all_corr.append(pd.DataFrame(np.array([beh_error,eye_error]).T,columns=['beh','eye']).corr().iloc[0,1])

        ax.plot(np.linspace(-10,10,100),np.linspace(-10,10,100),'k--')
        # style axis
        if s == 0:
            ax.set_ylabel('error gaze (dva)')
        else:
            ax.set_ylabel('')
        ax.spines['right'].set_visible(False)
        ax.spines['top'].set_visible(False)
        ax.set_xlabel('error behavior (dva)')
        ax.set_xlim(-10,10)
        ax.set_ylim(-10,10)

        # ax.set_title(f'{speed_titles[s]}\n({speed_labels[s]}per sec)')
    bf_out = calculate_bf(np.expand_dims(all_corr,axis=1))
    print(f'(c) : mean correlation behaviour and gaze error for speed {speed_labels[s]} = {np.mean(all_corr)}, BF = 10^{np.log10(np.array(bf_out))}')


#************************************
### SAVE
#************************************
plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp1b.png',dpi=300)
plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp1b.pdf')

        

(a): speed fast -- BFs 10^[-0.52855114  1.44575481  4.3091333   6.88279935  9.36326251 10.17967922
 10.76680332]
(a): speed medium -- BFs 10^[-0.30902155  6.29016268  8.91120227  9.3810847  10.93445287 12.03396189
 12.22764183]
(a): speed slow -- BFs 10^[ 0.84792827  5.39967607  8.47161081  8.80024323  9.77051186 10.01011727
 10.16616726]


/var/folders/c8/98wkm49s56x8ym14q0dgnn1xj153k_/T/ipykernel_60485/2061222140.py:133: RuntimeWarning: Mean of empty slice
  toplot.append(np.nanmean(sel_epo._data[:,x_idx],axis=0))


(c) : mean correlation behaviour and gaze error for speed 23.43° = 0.7160225067354212, BF = 10^[7.27484145]
(c) : mean correlation behaviour and gaze error for speed 10.04° = 0.7471958645456724, BF = 10^[6.81817625]
(c) : mean correlation behaviour and gaze error for speed 5.02° = 0.7661928931541773, BF = 10^[6.38443884]


### Second plot comparing different speeds directly

In [11]:
from scipy.stats import linregress
### SETUP
%matplotlib qt
plt.close('all')
fig,axs = plt.subplots(1,2,figsize=(a4_width,a4_height*0.35))

colors = sns.color_palette("rocket",3)
x = np.arange(7)
jitter_pos = [-0.2,0,0.2]


## Helper function
def draw_bracket(ax, x_pos, y1, y2, text, color='black', lw=1.5, bracket_width=-0.1, text_offset=0.15):
    y_mid = (y1 + y2) / 2
    ax.plot([x_pos, x_pos], [y1, y2], color=color, lw=lw, clip_on=False)
    ax.plot([x_pos, x_pos + bracket_width], [y1, y1], color=color, lw=lw, clip_on=False)
    ax.plot([x_pos, x_pos + bracket_width], [y2, y2], color=color, lw=lw, clip_on=False)
    ax.text(x_pos + bracket_width + text_offset, y_mid, text,
            va='center', ha='left', fontsize=6, color=color)
    
def y_at(x_pos, intercepts, slopes):
    return np.mean(intercepts) + np.mean(slopes) * x_pos

def get_bf_text(bf):
    if bf>6:
        return_text = 'BF>6'
    else: 
        return_text = f'BF={bf}'
    return return_text

#************************************
### (A) BEHAVIOUR 
#************************************
ax = axs[0]
group_error = []

all_slopes,all_intercepts,all_ys = [],[],[]
lines_toplot = []
for s in range(3):
    slopes,intercepts,ys,group_error_speed = [],[],[],[]

    for p in range(len(group_data[s])):
        y = np.array(group_data[s][p].stop_pos_deg-group_data[s][p].pos_pressed_deg)
        res = linregress(x, y)
        intercepts.append(res.intercept)
        slopes.append(res.slope)
        ys.append(y)
        ax.plot(x+jitter_pos[s],y,'.',color=subj_colors[s],alpha=0.3,markersize=8)
        group_error_speed.append(y)

    group_error.append(group_error_speed)
    lines_toplot.append(np.mean(intercepts) + np.mean(slopes)*x)

    all_slopes.append(slopes)
    all_intercepts.append(intercepts)
    all_ys.append(ys)
[ax.plot(x, line, label=f'{speed_titles[i]}\n({speed_labels[i]}per sec)',color=colors[i],lw=3) for i,line in enumerate(lines_toplot)]


ax.set_xticks(np.arange(7))
ax.set_xticklabels([str(np.round(i,2)) for i in group_data[s][p].stop_pos_deg.to_numpy()], fontsize=8)
ax.set_xlabel('stopping position (dva)')
ax.set_ylabel('Deviation from true stopping position (dva)')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.set_title('Behavior')
ax.legend(frameon=False,loc='upper left',fontsize=8)

group_error_beh = group_error.copy()



### add BFs
comparison = np.array(all_slopes)[1]-np.array(all_slopes)[0]
text0 = calculate_bf(np.expand_dims(comparison,axis=1))

comparison = np.array(all_slopes)[2]-np.array(all_slopes)[1]
text1 = calculate_bf(np.expand_dims(comparison,axis=1))

comparison = np.array(all_slopes)[2]-np.array(all_slopes)[0]
text2 = calculate_bf(np.expand_dims(comparison,axis=1))

x_bracket = x.max()

y0 = y_at(x_bracket, all_intercepts[0], all_slopes[0])
y1 = y_at(x_bracket, all_intercepts[1], all_slopes[1])
y2 = y_at(x_bracket, all_intercepts[2], all_slopes[2])

# Draw brackets for each pairwise comparison
draw_bracket(ax, x_bracket+0.4, y0+0.05, y1+0.05, f'{get_bf_text(int(text0[0]))}')
draw_bracket(ax, x_bracket+0.4, y1-+0.05, y2-+0.05, f'{get_bf_text(int(text1[0]))}')
draw_bracket(ax, x_bracket + 1.2, y0, y2, f'{get_bf_text(int(text2[0]))}')

print('behavior')
print(np.log10(text0),np.log10(text1),np.log10(text2))

#************************************
### (B) EYETRACKING 
#************************************
colors = sns.color_palette("mako",3)

group_error = []
ax = axs[1]
all_slopes,all_intercepts,all_ys = [],[],[]
lines_toplot = []
for s in range(3):
    slopes,intercepts,ys,group_error_speed = [],[],[],[]
    # x = np.sort(np.unique(all_epochs[0].metadata.loc[all_epochs[0].metadata.speed_idx==s,'stop_movement_time']))

    for p in range(np.array(group_final_gaze).shape[2]):
        y = np.array(np.array(group_data[s][p].stop_pos_deg)-np.array(group_final_gaze)[s,:,p])
        res = linregress(x, y)
        intercepts.append(res.intercept)
        slopes.append(res.slope)
        ys.append(y)
        ax.plot(x+jitter_pos[s],y,'.',color=colors[s],alpha=0.3,markersize=8)
        group_error_speed.append(y)

    group_error.append(group_error_speed)
    lines_toplot.append(np.mean(intercepts) + np.mean(slopes)*x)

    all_slopes.append(slopes)
    all_intercepts.append(intercepts)
    all_ys.append(ys)

[ax.plot(x, line,label= f'{speed_titles[i]}\n({speed_labels[i]}per sec)',color=colors[i],lw=3) for i,line in enumerate(lines_toplot)]


ax.set_xticks(np.arange(7))
ax.set_xticklabels([str(np.round(i,2)) for i in group_data[s][p].stop_pos_deg.to_numpy()],fontsize=8)
ax.set_xlabel('stopping position (dva)')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.set_title('Eyetracking')

ax.legend(frameon=False,loc='upper left',fontsize=8)

group_error_gaze = group_error.copy()



### add BFs
comparison = np.array(all_slopes)[1]-np.array(all_slopes)[0]
text0 = calculate_bf(np.expand_dims(comparison,axis=1))

comparison = np.array(all_slopes)[2]-np.array(all_slopes)[1]
text1 = calculate_bf(np.expand_dims(comparison,axis=1))

comparison = np.array(all_slopes)[2]-np.array(all_slopes)[0]
text2 = calculate_bf(np.expand_dims(comparison,axis=1))

x_bracket = x.max()


y0 = y_at(x_bracket, all_intercepts[0], all_slopes[0])
y1 = y_at(x_bracket, all_intercepts[1], all_slopes[1])
y2 = y_at(x_bracket, all_intercepts[2], all_slopes[2])

# Draw brackets for each pairwise comparison
draw_bracket(ax, x_bracket+0.4, y0+0.05, y1+0.05, f'{get_bf_text(int(text0[0]))}')
draw_bracket(ax, x_bracket+0.4, y1-+0.05, y2-+0.05, f'{get_bf_text(int(text1[0]))}')
draw_bracket(ax, x_bracket + 1.2, y0, y2, f'{get_bf_text(int(text2[0]))}')

print('gaze')
print(np.log10(text0),np.log10(text1),np.log10(text2))

#************************************
### SAVE
#************************************
plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp1b_slopes.png',dpi=300)
plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp1b_slopes.pdf')



behavior
[3.83650265] [4.60109682] [5.79861843]
gaze
[1.52131023] [0.1617848] [1.1295628]


### Third plot

Direct way of testing whether the time we have looking at the stimulus matters for how good the representation during occlusion is  

-- Sum the deviation between eye location & target at each timepoint all the way up until when occlusion starts <-> correlate that with the behavioural error at that time [do for 1a and 1b]


In [17]:
# get trajectory and timing for this current speed
plt.close('all')
fig,axs = plt.subplots(2,3,sharex=True,sharey=False,figsize=(a4_width,a4_height*0.5))

for i, (ax, label) in enumerate(zip([axs[0][0],axs[1][0]],['(a)','(b)'])):
    pos = ax.get_position()  
    x = pos.x0 - 0.11
    y = pos.y1 + 0.05
    fig.text(x, y, label, fontsize=12, fontweight='bold', va='top', ha='left')

corrs_all_speeds = []
for s in range(3):
    ax = axs[1][s]
    ax.set_ylim([-1,14])
    ax.set_ylabel('x-gaze error (dva)')
    ax_ind = axs[0][s]
    corrs = []
    ax_ind.set_title(f'{speed_titles[s]}\n({speed_labels[s]}per sec)')
    ax_ind.set_ylabel('x-gaze error (dva)')
    ax_ind.spines['top'].set_visible(False)
    ax_ind.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    for p in range(len(all_epochs)):
        epos = all_epochs[p].copy()
        epos_curr_speed = epos[epos.metadata['speed_idx']==s]

        x_idx = np.where([i=='xpos' for i in epos_curr_speed.ch_names])[0][0]
        epos_curr_speed._data[epos_curr_speed.metadata.direction==2,x_idx,:] = epos_curr_speed._data[epos_curr_speed.metadata.direction==2,x_idx,:]*-1


        traj = np.linspace(np.min(epos_curr_speed.metadata.start_pos_deg.unique()),np.max(epos_curr_speed.metadata.start_pos_deg.unique()),epos_curr_speed.metadata['n_frames'].unique()[0])
        total_traj_time = epos_curr_speed.metadata['n_frames'].unique()[0]*1/epos_curr_speed.metadata['monitor_refreshrate'].unique()[0]
        traj_tv = np.arange(0,total_traj_time,1/epos_curr_speed.metadata['monitor_refreshrate'].unique()[0])


        ## figure out which timepoints correspond to each other in eyetracking (defined by eyelink sampling rate) vs trajectory (defined by screen sampling rate)
        large_times = epos_curr_speed.times
        small_times = traj_tv

        indices = np.searchsorted(large_times, small_times)

        # correct edge cases
        indices = np.clip(indices, 1, len(large_times)-1)

        # choose nearest neighbor
        left = large_times[indices - 1]
        right = large_times[indices]

        indices -= (small_times - left) < (right - small_times)


        ## plot
        pre_occlusion_indices = traj_tv<epos_curr_speed.metadata.occlusion_start_time.unique()
        toplot = epos_curr_speed._data[:,x_idx,indices[pre_occlusion_indices]]-traj[pre_occlusion_indices]

        plot_tmp = ax.plot(traj_tv[pre_occlusion_indices],np.nanmean(toplot,axis=0),color=subj_colors[p])

        if (p == 5)or (p == 20) :
            [ax_ind.plot(traj_tv[pre_occlusion_indices],toplot[i,:],lw = 1, c=plot_tmp[-1].get_color(),alpha=0.15) for i in range(toplot.shape[0])]
            ax_ind.plot(traj_tv[pre_occlusion_indices],np.nanmean(toplot,axis=0),lw = 3, c=plot_tmp[-1].get_color(),label=f'sub-S{str(p+1).zfill(2)}')
            ax_ind.set_ylim([-5,25])
            ax_ind.legend(frameon=False)

        beh_deviation = np.array(epos_curr_speed.metadata.stop_pos_deg-epos_curr_speed.metadata.pos_pressed_deg)
        
        # # option 1: using summed deviation 
        # summed_eye_deviation = np.nansum(toplot,axis=1)
        # corrs.append(np.corrcoef(beh_deviation,summed_eye_deviation)[0,1])

        # option 2: using last timepoint before occlusion
        last_eye_deviation = toplot[:,-1]
        non_nan_indx = ~np.isnan(last_eye_deviation)

        corrs.append(np.corrcoef(last_eye_deviation[non_nan_indx],beh_deviation[non_nan_indx])[0,1])

    corrs_all_speeds.append(corrs)


fig.supxlabel('time from trial start until occlusion (s)')


# violin plot to show correlations
# remove NaNs from each subgroup
groups = [row[~np.isnan(row)] for row in np.array(corrs_all_speeds)]

for s in range(3):
    ax = axs[1][s]

    # create inset in top-right corner
    axins = inset_axes(ax, width="35%", height="35%", loc="upper right")

    vplot=axins.violinplot(groups[s], showmeans=True)

    for pc in vplot['bodies']:
        pc.set_facecolor('black')
        pc.set_edgecolor('black')
    for partname in ('cmeans', 'cmaxes', 'cmins', 'cbars'):
        if partname in vplot:
            vplot[partname].set_color('black')
            vplot[partname].set_linewidth(1.5)


    axins.set_xticks([])
    axins.set_ylabel('Correlation\ngaze<->behavior error',fontsize=7)
    axins.set_ylim([-0.7,0.7])

    axins.spines['top'].set_visible(False)
    axins.spines['right'].set_visible(False)


    bf_out = calculate_bf(np.expand_dims(groups[s],axis=1))
    print(f'mean correlation behaviour and gaze error for speed {speed_titles[s]} = {np.mean(groups[s])}, BF = 10^{np.log10(np.array(bf_out))}')


plt.tight_layout()


#************************************
### SAVE
#************************************
plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp1b_pre_occlusion.png',dpi=300)
plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp1b_pre_occlusion.pdf')


/var/folders/c8/98wkm49s56x8ym14q0dgnn1xj153k_/T/ipykernel_51501/570671234.py:58: RuntimeWarning: Mean of empty slice
  plot_tmp = ax.plot(traj_tv[pre_occlusion_indices],np.nanmean(toplot,axis=0),color=subj_colors[p])
/opt/anaconda3/envs/py-r-bayes/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:576: RuntimeWarning: Mean of empty slice
  avg = a.mean(axis, **keepdims_kw)
/opt/anaconda3/envs/py-r-bayes/lib/python3.14/site-packages/numpy/_core/_methods.py:134: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/opt/anaconda3/envs/py-r-bayes/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/opt/anaconda3/envs/py-r-bayes/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/opt/anaconda3/envs/py-r-bayes/lib/python3.14/site-packages/numpy/lib/_functio

mean correlation behaviour and gaze error for speed fast = 0.06475091213398929, BF = 10^[-0.437982]
mean correlation behaviour and gaze error for speed medium = 0.13446484592518845, BF = 10^[-0.06678293]
mean correlation behaviour and gaze error for speed slow = 0.1037165030154059, BF = 10^[-0.26973298]


/var/folders/c8/98wkm49s56x8ym14q0dgnn1xj153k_/T/ipykernel_51501/570671234.py:117: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


### Bayes Factor Table

In [13]:
def make_table_grouped(data_combined, conditions, row_labels, unique_stop_movement_time, fn):
    from matplotlib.patches import Rectangle

    n_stops = len(unique_stop_movement_time)
    n_conds = len(conditions)

    sub_col_labels = list(conditions) * n_stops
    group_bg  = ['#DDEEFF', '#F5F5F5']
    group_hdr = ['#99BBDD', '#BBBBBB']

    fig, ax = plt.subplots(figsize=(a4_width, a4_height * 0.5))
    # fig, ax = plt.subplots(figsize=(a4_height, a4_width * 0.8)) # landscape
    fig.patch.set_visible(False)
    ax.axis('off')

    table = ax.table(
        cellText=data_combined,
        colLabels=sub_col_labels,
        rowLabels=list(row_labels),
        loc='center',
        cellLoc='center'
    )
    table.auto_set_font_size(False)
    table.set_fontsize(7)
    table.scale(1, 2)

    for col_idx in range(n_stops * n_conds):
        s = col_idx // n_conds
        # table[0, col_idx].set_facecolor(group_hdr[s % 2])
        for row_idx in range(1, len(row_labels) + 1):
            # table[row_idx, col_idx].set_facecolor(group_bg[s % 2])
            table[0, col_idx].get_text().set_fontweight('bold')
            table[0, col_idx].get_text().set_rotation(90)  
            table[0, col_idx].set_height(0.18)

    # ── draw super-headers as patches using actual rendered positions ──────────
    plt.tight_layout()
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    ax_inv   = ax.transAxes.inverted()

    for s in range(n_stops):
        cell_first = table[0, s * n_conds]
        cell_last  = table[0, (s + 1) * n_conds - 1]

        bf = cell_first.get_window_extent(renderer)
        bl = cell_last.get_window_extent(renderer)

        # span from left edge of first sub-col to right edge of last sub-col
        # and sit one cell-height above the sub-header row
        p0 = ax_inv.transform([bf.x0, bf.y1])
        p1 = ax_inv.transform([bl.x1, bf.y1 + bf.height])

        rect = Rectangle(
            (p0[0], p0[1]), p1[0] - p0[0], p1[1] - p0[1],
            facecolor='white', edgecolor='black', linewidth=1,
            transform=ax.transAxes, clip_on=False
        )
        ax.add_patch(rect)

        ax.text(
            (p0[0] + p1[0]) / 2, (p0[1] + p1[1]) / 2,
            f"stop {s}\n {unique_stop_movement_time[s]}",
            ha='center', va='center', fontsize=10, fontweight='bold',
            transform=ax.transAxes, clip_on=False
        )

    plt.savefig(f"{bids_dir}/{task}/derivatives/figures/{fn}.pdf", bbox_inches='tight')





### Make two tables, one for the behavioural error with BFs and one for gaze error
## Behavioural error 
n_subjects = np.array(group_error_beh).shape[1]
n_stops = np.array(group_error_beh).shape[2]
n_conds = np.array(group_error_beh).shape[0]

# instead of pure errors per condition, calculate a differences between each set of two
combinations= [[0,1],[1,2],[0,2]]
conditions,conditions2 = [],[]
beh_error_difference_by_cond = np.zeros((n_conds,n_stops,n_subjects))
for ci,c in enumerate(combinations):
    conditions.append(f'{speed_labels[c[0]]}-{speed_labels[c[1]]}')
    conditions2.append(f'{speed_titles[c[0]]}-{speed_titles[c[1]]}')
    beh_error_difference_by_cond[ci,:,:] = (np.array(group_error_beh)[c[0],:,:]-np.array(group_error_beh)[c[1],:,:]).reshape(-1,n_subjects)

rows_per_cond = []
for ge in beh_error_difference_by_cond:
    arr       = np.array(ge)
    mean_err  = np.round(np.mean(arr, axis=1), 2)
    bf        = calculate_bf(arr.T)
    bf_str    = np.array([f"$10^{{{x:.2f}}}$" for x in np.log10(bf)])
    rows_per_cond.append(np.vstack([mean_err, bf_str]))  # (2, n_stops)


# Interleave: for each stop, stack all conditions side by side
data_combined = np.hstack([
    np.column_stack([rows_per_cond[c][:, s] for c in range(n_conds)])
    for s in range(n_stops)
])  # shape → (2, n_stops * n_conds)

make_table_grouped(
    data_combined=data_combined,
    conditions=conditions2,
    row_labels=['behavioral error (dva)', 'Bayes Factor'],
    unique_stop_movement_time=[f"{str(np.round(i,2))}\u00B0" for i in group_data[s][p].stop_pos_deg.to_numpy()],
    fn='bfs_beh_error'
)



## Gaze error 
n_subjects = np.array(group_error_gaze).shape[1]
n_stops = np.array(group_error_gaze).shape[2]
n_conds = np.array(group_error_gaze).shape[0]

# instead of pure errors per condition, calculate a differences between each set of two
combinations= [[0,1],[1,2],[0,2]]
conditions = []
gaze_error_difference_by_cond = np.zeros((n_conds,n_stops,n_subjects))
for ci,c in enumerate(combinations):
    conditions.append(f'{speed_labels[c[0]]}-{speed_labels[c[1]]}')
    gaze_error_difference_by_cond[ci,:,:] = (np.array(group_error_gaze)[c[0],:,:]-np.array(group_error_gaze)[c[1],:,:]).reshape(-1,n_subjects)

rows_per_cond = []
for ge in gaze_error_difference_by_cond:
    arr       = np.array(ge)
    mean_err  = np.round(np.mean(arr, axis=1), 2)
    bf        = calculate_bf(arr.T)
    bf_str    = np.array([f"$10^{{{x:.2f}}}$" for x in np.log10(bf)])
    rows_per_cond.append(np.vstack([mean_err, bf_str]))  # (2, n_stops)


# Interleave: for each stop, stack all conditions side by side
data_combined = np.hstack([
    np.column_stack([rows_per_cond[c][:, s] for c in range(n_conds)])
    for s in range(n_stops)
])  # shape → (2, n_stops * n_conds)

make_table_grouped(
    data_combined=data_combined,
    conditions=conditions2,
    row_labels=['gaze error (dva)', 'Bayes Factor'],
    unique_stop_movement_time=[f"{str(np.round(i,2))}\u00B0" for i in group_data[s][p].stop_pos_deg.to_numpy()],
    fn='bfs_gaze_error'
)



